# TimeMixer -- ETTh1 Training

Reference: Wang et al., *TimeMixer: Decomposable Multiscale Mixing for Time Series
Forecasting*, ICLR 2024. https://arxiv.org/abs/2405.14616

Trains TimeMixer at four forecast horizons (96, 192, 336, 720) on ETTh1 (multivariate,
seq_len=512). Target: averaged MSE ~0.446, MAE ~0.434 (Wang et al., Table 1).

All model and dataset code is inlined so this notebook runs without a local package install.

In [1]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## Dataset

In [3]:
class ETTh1Dataset(Dataset):
    """ETTh1 multivariate dataset with chronological train/val/test split.

    Split follows the protocol used in PatchTST and TimeMixer papers:
        train: first 8640 rows  (12 months)
        val:   next  2880 rows  (4 months)
        test:  final 2880 rows  (4 months)

    Normalization: z-score per channel, fit on train split only.
    """

    SPLIT_SIZES = {"train": 8640, "val": 2880, "test": 2880}
    TARGET_COLS = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]

    def __init__(self, csv_path: str, split: str, seq_len: int, pred_len: int) -> None:
        if split not in self.SPLIT_SIZES:
            raise ValueError(f"split must be one of {list(self.SPLIT_SIZES)}, got '{split}'.")

        df = pd.read_csv(csv_path)[self.TARGET_COLS].values.astype(np.float32)

        train_end = self.SPLIT_SIZES["train"]
        val_end = train_end + self.SPLIT_SIZES["val"]

        train_data = df[:train_end]
        self._mean = train_data.mean(axis=0)
        self._std = train_data.std(axis=0)
        self._std = np.where(self._std == 0, 1.0, self._std)

        data = (df - self._mean) / self._std

        if split == "train":
            self._data = data[:train_end]
        elif split == "val":
            self._data = data[train_end:val_end]
        else:
            self._data = data[val_end:]

        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int):
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)

## Model

In [4]:
class SeriesDecomposition(nn.Module):
    """Trend via moving average; seasonal = series minus trend."""

    def __init__(self, kernel_size: int) -> None:
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.avg_pool = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x: torch.Tensor):
        x_padded = F.pad(x, (self.padding, self.padding), mode="reflect")
        trend = self.avg_pool(x_padded)
        if trend.shape[-1] != x.shape[-1]:
            trend = trend[..., : x.shape[-1]]
        seasonal = x - trend
        return seasonal, trend


class PDMBlock(nn.Module):
    """Past-Decomposable-Mixing: project seasonal + trend to d_model."""

    def __init__(self, seq_len: int, d_model: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.seasonal_mix = nn.Linear(seq_len, d_model)
        self.trend_mix = nn.Linear(seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, seasonal: torch.Tensor, trend: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.seasonal_mix(seasonal) + self.trend_mix(trend))


class FMMBlock(nn.Module):
    """Future-Multipredictor-Mixing: forecast from each scale, ensemble by learned weights."""

    def __init__(self, num_scales: int, d_model: int, pred_len: int) -> None:
        super().__init__()
        self.heads = nn.ModuleList([nn.Linear(d_model, pred_len) for _ in range(num_scales)])
        self.scale_weights = nn.Parameter(torch.ones(num_scales))

    def forward(self, scale_representations: list) -> torch.Tensor:
        weights = F.softmax(self.scale_weights, dim=0)
        forecasts = torch.stack(
            [w * head(rep) for w, head, rep in zip(weights, self.heads, scale_representations)],
            dim=0,
        )
        return forecasts.sum(dim=0)


class TimeMixer(nn.Module):
    """TimeMixer: decomposable multiscale mixing for multivariate forecasting."""

    def __init__(
        self,
        seq_len: int,
        pred_len: int,
        num_scales: int = 3,
        d_model: int = 16,
        decomp_kernel: int = 25,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.num_scales = num_scales
        self.scale_lens = [max(1, seq_len // (2**s)) for s in range(num_scales)]

        self.downsamplers = nn.ModuleList(
            [nn.Identity() if s == 0 else nn.AvgPool1d(kernel_size=2**s, stride=2**s) for s in range(num_scales)]
        )
        self.decomposition = SeriesDecomposition(decomp_kernel)
        self.pdm_blocks = nn.ModuleList(
            [PDMBlock(seq_len=l, d_model=d_model, dropout=dropout) for l in self.scale_lens]
        )
        self.fmm = FMMBlock(num_scales=num_scales, d_model=d_model, pred_len=pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)  # (B, C, seq_len)
        scale_reps = []
        for downsampler, pdm in zip(self.downsamplers, self.pdm_blocks):
            x_s = downsampler(x)
            seasonal, trend = self.decomposition(x_s)
            scale_reps.append(pdm(seasonal, trend))
        return self.fmm(scale_reps).transpose(1, 2)  # (B, pred_len, C)

## Training Infrastructure

In [5]:
class EarlyStopping:
    """Stop training when validation MSE has not improved for `patience` epochs."""

    def __init__(self, patience: int = 10, checkpoint_path: str = "best_model.pt") -> None:
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_val_mse = float("inf")
        self.counter = 0
        self.best_epoch = 0

    def step(self, val_mse: float, model: nn.Module, epoch: int) -> bool:
        if val_mse < self.best_val_mse:
            self.best_val_mse = val_mse
            self.counter = 0
            self.best_epoch = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple:
    mse = torch.mean((pred - target) ** 2).item()
    mae = torch.mean(torch.abs(pred - target)).item()
    return mse, mae


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        mse, mae = compute_metrics(pred.detach(), y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        mse, mae = compute_metrics(pred, y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n

## Config and Paths

In [9]:
CSV_PATH = "/kaggle/input/datasets/alaaelmor/ettsmall/ETTh1.csv"
RESULTS_DIR = Path("results")
CKPT_DIR = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seq_len": 512,
    "num_scales": 3,
    "d_model": 16,
    "decomp_kernel": 25,
    "dropout": 0.1,
    "lr": {96: 1e-3, 192: 5e-4, 336: 5e-4, 720: 1e-4},
    "batch_size": 32,
    "epochs": 100,
    "patience": 10,
    "seed": SEED,
}

HORIZONS = [96, 192, 336, 720]

# Published targets from Wang et al., ICLR 2024, Table 1 (ETTh1, avg over horizons).
PAPER_AVG_MSE = 0.446
PAPER_AVG_MAE = 0.434

## Training Loop

In [10]:
all_results = []

for pred_len in HORIZONS:
    print(f'\n{"=" * 60}')
    print(f"Horizon: pred_len={pred_len}")
    print(f'{"=" * 60}')

    torch.manual_seed(CONFIG["seed"])
    random.seed(CONFIG["seed"])
    np.random.seed(CONFIG["seed"])

    train_ds = ETTh1Dataset(CSV_PATH, "train", CONFIG["seq_len"], pred_len)
    val_ds = ETTh1Dataset(CSV_PATH, "val", CONFIG["seq_len"], pred_len)
    test_ds = ETTh1Dataset(CSV_PATH, "test", CONFIG["seq_len"], pred_len)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    model = TimeMixer(
        seq_len=CONFIG["seq_len"],
        pred_len=pred_len,
        num_scales=CONFIG["num_scales"],
        d_model=CONFIG["d_model"],
        decomp_kernel=CONFIG["decomp_kernel"],
        dropout=CONFIG["dropout"],
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"][pred_len])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
    criterion = nn.MSELoss()

    ckpt_path = str(CKPT_DIR / f"timemixer_etth1_pred{pred_len}.pt")
    early_stopping = EarlyStopping(patience=CONFIG["patience"], checkpoint_path=ckpt_path)

    log_rows = []
    t0 = time.time()

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_mse, train_mae = train_one_epoch(model, train_loader, optimizer, criterion)
        val_mse, val_mae = evaluate(model, val_loader)
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        log_rows.append(
            {
                "epoch": epoch,
                "train_mse": round(train_mse, 6),
                "train_mae": round(train_mae, 6),
                "val_mse": round(val_mse, 6),
                "val_mae": round(val_mae, 6),
                "lr": lr_now,
            }
        )

        if epoch % 10 == 0 or epoch == 1:
            elapsed = time.time() - t0
            print(
                f'  Epoch {epoch:3d}/{CONFIG["epochs"]} | '
                f"train MSE {train_mse:.4f} | val MSE {val_mse:.4f} | "
                f"lr {lr_now:.2e} | {elapsed:.0f}s"
            )

        if early_stopping.step(val_mse, model, epoch):
            print(f"  Early stop at epoch {epoch}. Best epoch: {early_stopping.best_epoch}.")
            break

    log_path = RESULTS_DIR / f"timemixer_etth1_pred{pred_len}_log.csv"
    pd.DataFrame(log_rows).to_csv(log_path, index=False)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_mse, test_mae = evaluate(model, test_loader)

    print(
        f"  Test MSE: {test_mse:.4f} | Test MAE: {test_mae:.4f} | "
        f"Best val MSE: {early_stopping.best_val_mse:.4f} @ epoch {early_stopping.best_epoch}"
    )

    all_results.append(
        {
            "model": "TimeMixer",
            "dataset": "ETTh1",
            "seq_len": CONFIG["seq_len"],
            "pred_len": pred_len,
            "test_mse": round(test_mse, 6),
            "test_mae": round(test_mae, 6),
            "best_val_mse": round(early_stopping.best_val_mse, 6),
            "best_epoch": early_stopping.best_epoch,
            "num_params": total_params,
            "seed": CONFIG["seed"],
        }
    )

results_df = pd.DataFrame(all_results)
results_path = RESULTS_DIR / "timemixer_etth1.csv"
results_df.to_csv(results_path, index=False)
print(f"\nResults saved to {results_path}")
print(results_df[["pred_len", "test_mse", "test_mae"]].to_string(index=False))


Horizon: pred_len=96
Parameters: 33,667
  Epoch   1/100 | train MSE 0.4491 | val MSE 0.6991 | lr 1.00e-03 | 2s
  Epoch  10/100 | train MSE 0.3474 | val MSE 0.6884 | lr 9.76e-04 | 18s
  Epoch  20/100 | train MSE 0.3447 | val MSE 0.6733 | lr 9.05e-04 | 37s
  Early stop at epoch 27. Best epoch: 17.
  Test MSE: 0.4539 | Test MAE: 0.4524 | Best val MSE: 0.6574 @ epoch 17

Horizon: pred_len=192
Parameters: 38,563
  Epoch   1/100 | train MSE 0.5602 | val MSE 0.9184 | lr 5.00e-04 | 2s
  Epoch  10/100 | train MSE 0.3993 | val MSE 0.8827 | lr 4.88e-04 | 19s
  Epoch  20/100 | train MSE 0.3917 | val MSE 0.9147 | lr 4.52e-04 | 37s
  Early stop at epoch 21. Best epoch: 11.
  Test MSE: 0.4990 | Test MAE: 0.4814 | Best val MSE: 0.8763 @ epoch 11

Horizon: pred_len=336
Parameters: 45,907
  Epoch   1/100 | train MSE 0.6135 | val MSE 1.0466 | lr 5.00e-04 | 2s
  Epoch  10/100 | train MSE 0.4432 | val MSE 0.9861 | lr 4.88e-04 | 18s
  Epoch  20/100 | train MSE 0.4353 | val MSE 0.9936 | lr 4.52e-04 | 36s
  

## Summary and Benchmark Comparison

In [11]:
avg_mse = results_df["test_mse"].mean()
avg_mae = results_df["test_mae"].mean()

print("=== TimeMixer ETTh1 Results ===")
print(results_df[["pred_len", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))
print(f"\nAverage | MSE {avg_mse:.4f} | MAE {avg_mae:.4f}")
print(f"Paper   | MSE {PAPER_AVG_MSE:.4f} | MAE {PAPER_AVG_MAE:.4f}")
print(f"Gap     | MSE {avg_mse - PAPER_AVG_MSE:+.4f} | MAE {avg_mae - PAPER_AVG_MAE:+.4f}")

=== TimeMixer ETTh1 Results ===
 pred_len  test_mse  test_mae  best_epoch
       96  0.453905  0.452372          17
      192  0.498951  0.481448          11
      336  0.543081  0.514292          16
      720  0.675258  0.601949          10

Average | MSE 0.5428 | MAE 0.5125
Paper   | MSE 0.4460 | MAE 0.4340
Gap     | MSE +0.0968 | MAE +0.0785


## Verify Output Files

In [12]:
required = [RESULTS_DIR / "timemixer_etth1.csv"] + [
    RESULTS_DIR / f"timemixer_etth1_pred{h}_log.csv" for h in HORIZONS
]

all_present = True
for path in required:
    exists = path.exists()
    print(f'  [{"OK" if exists else "MISSING"}] {path}')
    if not exists:
        all_present = False

if not all_present:
    raise RuntimeError("Some output files are missing. Do not close the session.")
print("\nAll output files verified.")

  [OK] results/timemixer_ettch1.csv
  [OK] results/timemixer_ettch1_pred96_log.csv
  [OK] results/timemixer_ettch1_pred192_log.csv
  [OK] results/timemixer_ettch1_pred336_log.csv
  [OK] results/timemixer_ettch1_pred720_log.csv

All output files verified.
